# Plant Disease Detector - Model Training
Transfer learning with **MobileNetV2** on the **PlantVillage** dataset (38 classes, ~54k leaf images).

**Before running:** in Colab go to *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*.
Total training time is roughly 15-25 minutes on a free T4 GPU.

## 1. Setup

In [ ]:
!pip -q install datasets

import json, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
tf.keras.utils.set_random_seed(SEED)

## 2. Load the dataset and split it
PlantVillage (38 classes, ~54k leaf images) from the Hugging Face Hub. The download is automatic and takes a few minutes the first time.
The dataset's own train/test split is used as **train / test**, and 10% of train is held out as **validation**.

In [ ]:
from datasets import load_dataset, ClassLabel, Image as HFImage

raw = load_dataset("BrandonFors/Plant-Diseases-PlantVillage-Dataset")
print(raw)

feats = raw["train"].features
lbl_col = [k for k, v in feats.items() if isinstance(v, ClassLabel)][0]
img_col = [k for k, v in feats.items() if isinstance(v, HFImage)][0]

class_names = feats[lbl_col].names
NUM_CLASSES = len(class_names)

if "test" in raw:
    tv = raw["train"].train_test_split(test_size=0.1, seed=SEED, stratify_by_column=lbl_col)
    train_hf, val_hf, test_hf = tv["train"], tv["test"], raw["test"]
else:
    a = raw["train"].train_test_split(test_size=0.2, seed=SEED, stratify_by_column=lbl_col)
    b = a["test"].train_test_split(test_size=0.5, seed=SEED, stratify_by_column=lbl_col)
    train_hf, val_hf, test_hf = a["train"], b["train"], b["test"]

def to_tf(hf_ds):
    def gen():
        for ex in hf_ds:
            yield np.array(ex[img_col].convert("RGB"), dtype=np.uint8), ex[lbl_col]
    return tf.data.Dataset.from_generator(
        gen, output_signature=(tf.TensorSpec((None, None, 3), tf.uint8),
                               tf.TensorSpec((), tf.int64)))

train_raw, val_raw, test_raw = to_tf(train_hf), to_tf(val_hf), to_tf(test_hf)

print("Classes:", NUM_CLASSES)
print("Train / Val / Test:", len(train_hf), len(val_hf), len(test_hf))
print(class_names[:5], "...")

## 3. Preprocessing (resize to 224x224) and input pipeline

In [ ]:
def resize(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return tf.cast(image, tf.float32), label

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (train_raw.map(resize, AUTOTUNE).shuffle(2000, seed=SEED)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds   = val_raw.map(resize, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds  = test_raw.map(resize, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Quick look at some samples
plt.figure(figsize=(12, 6))
for i, (img, lbl) in enumerate(train_ds.unbatch().take(8)):
    plt.subplot(2, 4, i + 1)
    plt.imshow(img.numpy().astype("uint8"))
    plt.title(class_names[lbl.numpy()].replace("___", "\n"), fontsize=8)
    plt.axis("off")
plt.tight_layout(); plt.show()

## 4. Model: data augmentation + MobileNetV2 (transfer learning)
Augmentation (rotation, flips, zoom, brightness) is built into the model and only runs during training.
Pixel values are scaled by the model's `Rescaling` layer to the range MobileNetV2 expects (-1 to 1, a normalised 0-1 style scaling).

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
], name="augmentation")

base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base_model.trainable = False   # phase 1: freeze pretrained layers

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1 / 127.5, offset=-1)(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

## 5. Training
**Phase 1** trains only the new classification head. **Phase 2** fine-tunes the top layers of MobileNetV2 with a much lower learning rate.
Loss is categorical cross-entropy (the `sparse_` version, which is the same loss but takes integer labels), optimizer is Adam, and early stopping watches validation loss.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True)

# Phase 1: train the head
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history1 = model.fit(train_ds, validation_data=val_ds,
                     epochs=8, callbacks=[early_stop])

In [ ]:
# Phase 2: fine-tune the top of MobileNetV2
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history2 = model.fit(train_ds, validation_data=val_ds,
                     epochs=10, callbacks=[early_stop])

## 6. Training curves

In [ ]:
acc     = history1.history["accuracy"]     + history2.history["accuracy"]
val_acc = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss     = history1.history["loss"]     + history2.history["loss"]
val_loss = history1.history["val_loss"] + history2.history["val_loss"]
split = len(history1.history["accuracy"])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(acc, label="train"); ax[0].plot(val_acc, label="validation")
ax[0].axvline(split - 1, ls="--", c="gray", label="start fine-tuning")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(loss, label="train"); ax[1].plot(val_loss, label="validation")
ax[1].axvline(split - 1, ls="--", c="gray")
ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

## 7. Evaluation on the test set (accuracy, precision, recall, F1, confusion matrix)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest accuracy: {test_acc*100:.2f}%   <-- put this in [X]% in your report")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds), axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(16, 14))
plt.imshow(cm_norm, cmap="Blues")
plt.colorbar(fraction=0.03)
plt.xticks(range(NUM_CLASSES), class_names, rotation=90, fontsize=7)
plt.yticks(range(NUM_CLASSES), class_names, fontsize=7)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Normalized confusion matrix")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

## 8. Save the model (Keras + TensorFlow Lite for mobile) and class names

In [ ]:
model.save("plant_disease_model.keras")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
with open("plant_disease_model.tflite", "wb") as f:
    f.write(converter.convert())

with open("class_names.json", "w") as f:
    json.dump(class_names, f)

import os
print("Keras size  (MB):", round(os.path.getsize("plant_disease_model.keras")/1e6, 1))
print("TFLite size (MB):", round(os.path.getsize("plant_disease_model.tflite")/1e6, 1))

## 9. Try it on your own leaf photo

In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image as kimage

def predict_leaf(path, top_k=3):
    img = kimage.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = np.expand_dims(kimage.img_to_array(img), 0)
    probs = model.predict(arr, verbose=0)[0]
    plt.imshow(img); plt.axis("off"); plt.show()
    for i in np.argsort(probs)[::-1][:top_k]:
        print(f"{class_names[i]:50s} {probs[i]*100:5.1f}%")

uploaded = files.upload()
for name in uploaded:
    predict_leaf(name)

## 10. Download the outputs

In [ ]:
for f in ["plant_disease_model.keras", "plant_disease_model.tflite",
          "class_names.json", "training_curves.png", "confusion_matrix.png"]:
    files.download(f)